# **I. Tiền xử lý chung**

**1. Kiểm tra dữ liệu gốc**

In [1]:
from pathlib import Path
import pandas as pd
import sys
from IPython.display import display

PROJECT_ROOT = (
    Path.cwd()
    if (Path.cwd() / "data/raw").exists()
    else Path.cwd().parent
)

# Import module dùng chung khi chạy từ project hoặc notebooks.
project_path = str(PROJECT_ROOT.resolve())
if project_path not in sys.path:
    sys.path.insert(0, project_path)

import src.preprocessing as preprocessing

apps_raw = pd.read_csv(
    PROJECT_ROOT / "data/raw/googleplaystore.csv"
)

apps = apps_raw.copy()

print(f"Số dòng ban đầu: {len(apps):,}")

Số dòng ban đầu: 10,841


**2. Kiểm tra và loại bản ghi lệch cột**  

Nhóm loại bản ghi có dấu hiệu lệch cột vì nhiều giá trị không phù hợp với ý nghĩa của thuộc tính và chưa có căn cứ khôi phục chính xác. Sau đó, nhóm loại các dòng trùng lặp hoàn toàn, giữ lần xuất hiện đầu tiên. Các bản ghi chỉ trùng tên ứng dụng được giữ lại để xem xét theo từng câu hỏi phân tích.

In [2]:
malformed_mask = preprocessing.malformed_app_mask(apps)

print("Các bản ghi có dấu hiệu lệch cột:")
display(apps.loc[malformed_mask])

rows_before = len(apps)
apps = apps.loc[~malformed_mask].copy()

print(f"Số dòng trước: {rows_before:,}")
print(f"Số dòng bị loại: {rows_before - len(apps):,}")
print(f"Số dòng sau: {len(apps):,}")

Các bản ghi có dấu hiệu lệch cột:


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


Số dòng trước: 10,841
Số dòng bị loại: 1
Số dòng sau: 10,840


Nhóm xác định và loại một bản ghi của ứng dụng Life Made WI-Fi Touchscreen Photo Frame, có nhiều giá trị không phù hợp với ý nghĩa của cột. Số dòng giảm từ 10.841 xuống 10.840.

**3. Loại bản sao hoàn toàn**  



In [3]:
rows_before = len(apps)
duplicate_count = apps.duplicated().sum()

apps = apps.drop_duplicates(keep="first").copy()

print(f"Số dòng trước: {rows_before:,}")
print(f"Số bản sao hoàn toàn bị loại: {duplicate_count:,}")
print(f"Số dòng sau: {len(apps):,}")
print(f"Số bản sao hoàn toàn còn lại: {apps.duplicated().sum():,}")

Số dòng trước: 10,840
Số bản sao hoàn toàn bị loại: 483
Số dòng sau: 10,357
Số bản sao hoàn toàn còn lại: 0


Nhóm loại 483 bản sao giống nhau hoàn toàn, giữ lại bản ghi xuất hiện đầu tiên. Bảng còn 10.357 dòng và không còn dòng trùng lặp hoàn toàn.

**4. Kiểm tra các bản ghi trùng tên còn lại**

In [4]:
repeated_names = apps["App"].duplicated(keep=False)

display(pd.Series({
    "Remaining rows": len(apps),
    "Distinct app names": apps["App"].nunique(),
    "Extra rows with repeated app names": apps["App"].duplicated().sum(),
    "App names appearing more than once": (
        apps.loc[repeated_names, "App"].nunique()
    )
}))

Remaining rows                        10357
Distinct app names                     9659
Extra rows with repeated app names      698
App names appearing more than once      523
dtype: int64

Bảng còn 9.659 tên ứng dụng phân biệt. Trong đó, 523 tên xuất hiện nhiều lần, tạo ra 698 bản ghi thêm ngoài lần xuất hiện đầu tiên. Nhóm chưa loại các bản ghi này vì cùng tên không đồng nghĩa với thông tin hoàn toàn giống nhau; cách xử lý sẽ được xác định theo đơn vị phân tích của từng câu hỏi.

**5. Chuyển đổi các cột số**  

Nhóm sẽ thực hiện chuyển đổi các cột số như sau:  
- Giữ nguyên cột `Rating` vì đã có kiểu float64
- `Reviews` và `Installs` sang kiểu Int64
- `Price` sang float64  
- Giữ cột `Size` gốc dưới dạng `string`, thêm `Size_MB` kiểu `float64` để tính toán và `Size_varies` kiểu `boolean` để nhận diện placeholder  

Trước khi chuyển đổi, nhóm loại dấu phẩy và dấu + trong `Installs`, cùng ký hiệu $ trong `Price`. Giá trị `Installs` sau chuyển đổi vẫn là cận dưới của lượt cài đặt.

In [5]:
rows_before = len(apps)

# ---------- 1. Rating: giữ giá trị thiếu, chuyển sang float64 ----------
rating = pd.to_numeric(
    apps["Rating"], errors="raise"
).astype("float64")

# ---------- 2. Reviews: chuyển sang Int64 ----------
reviews = pd.to_numeric(
    apps["Reviews"], errors="raise"
).astype("Int64")

# ---------- 3. Installs: bỏ ký hiệu, giữ ý nghĩa cận dưới ----------
installs = pd.to_numeric(
    apps["Installs"].astype("string")
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False),
    errors="raise"
).astype("Int64")

# ---------- 4. Price: bỏ ký hiệu $, chuyển sang float64 ----------
price = pd.to_numeric(
    apps["Price"].astype("string")
    .str.replace("$", "", regex=False),
    errors="raise"
).astype("float64")

# ---------- 5. Size: quy đổi về MB, tách placeholder ----------
size_text = apps["Size"].astype("string").str.strip()

size_varies = size_text.eq("Varies with device").fillna(False)
size_missing = (
    size_text.isna()
    | size_text.eq("").fillna(False)
)

size_parts = size_text.str.extract(r"^(\d+(?:\.\d+)?)([Mk])$")

unexpected_size = (
    ~size_missing
    & ~size_varies
    & size_parts[0].isna()
)

if unexpected_size.any():
    display(apps.loc[unexpected_size, ["App", "Size"]])
    raise ValueError("Còn giá trị Size chưa xác định cách chuyển đổi.")

# Theo quy ước đã dùng ở notebook 1: k / 1024.
size_mb = (
    pd.to_numeric(size_parts[0], errors="raise")
    * size_parts[1].map({"M": 1.0, "k": 1 / 1024})
).astype("float64")

# ---------- 6. Cập nhật bảng sau khi các phép chuyển đổi thành công ----------
apps = apps.assign(
    Rating=rating,
    Reviews=reviews,
    Installs=installs,
    Price=price,
    Size_MB=size_mb,
    Size_varies=size_varies
)

# ---------- 7. Kiểm tra đủ năm cột số ----------
numeric_cols = ["Rating", "Reviews", "Size_MB", "Installs", "Price"]

conversion_summary = pd.DataFrame({
    "Dtype": apps[numeric_cols].dtypes.astype(str),
    "Missing count": apps[numeric_cols].isna().sum(),
    "Negative count": apps[numeric_cols].lt(0).sum(),
    "Min": apps[numeric_cols].min(),
    "Max": apps[numeric_cols].max()
})

print(f"Số dòng trước: {rows_before:,}")
print(f"Số dòng sau: {len(apps):,}")
display(conversion_summary)

invalid_rating = (
    apps["Rating"].notna()
    & ~apps["Rating"].between(1, 5)
)

print(f"Rating ngoài thang 1–5: {invalid_rating.sum():,}")
print(f"Size gốc bị thiếu: {size_missing.sum():,}")
print(f"Size là 'Varies with device': {size_varies.sum():,}")
print(f"Size_MB nhỏ hơn hoặc bằng 0: {apps['Size_MB'].le(0).sum():,}")

display(
    apps[
        ["App", "Rating", "Reviews", "Size", "Size_MB",
         "Size_varies", "Installs", "Price"]
    ].head()
)

Số dòng trước: 10,357
Số dòng sau: 10,357


,Dtype,Missing count,Negative count,Min,Max
Rating,float64,1465,0,1.0,5.0
Reviews,Int64,0,0,0.0,78158306.0
Size_MB,float64,1526,0,0.008301,100.0
Installs,Int64,0,0,0.0,1000000000.0
Price,float64,0,0,0.0,400.0


Rating ngoài thang 1–5: 0
Size gốc bị thiếu: 0
Size là 'Varies with device': 1,526
Size_MB nhỏ hơn hoặc bằng 0: 0


,App,Rating,Reviews,Size,Size_MB,Size_varies,Installs,Price
0,Photo Editor & Candy Camera & Grid & ScrapBook,4.1,159,19M,19.0,False,10000,0.0
1,Coloring book moana,3.9,967,14M,14.0,False,500000,0.0
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",4.7,87510,8.7M,8.7,False,5000000,0.0
3,Sketch - Draw & Paint,4.5,215644,25M,25.0,False,50000000,0.0
4,Pixel Draw - Number Art Coloring Book,4.3,967,2.8M,2.8,False,100000,0.0


**Nhận xét:**  
- Các cột số đã có kiểu dữ liệu đúng như dự kiến. `Reviews`, `Installs` và `Price` không có giá trị thiếu hoặc âm   
- `Rating` còn 1.465 giá trị thiếu; các điểm quan sát được đều nằm trong thang 1–5  
- `Size_MB` có 1.526 giá trị NaN, tương ứng hoàn toàn với các bản ghi Varies with device. Cột `Size` gốc không thiếu, cờ `Size_varies` giúp nhận diện các trường hợp kích thước chưa xác định  
- Không phát hiện kích thước nhỏ hơn hoặc bằng 0. Bước chuyển đổi không loại thêm bản ghi  

**6. Chuyển đổi cột thời gian `Last Updated`**  

Nhóm chuyển kiểu dữ liệu cột `Last Updated` từ chuỗi sang kiểu `datetime64[ns]` để hỗ trợ so sánh và trích xuất thông tin ngày tháng. Cột này biểu diễn lần cập nhật cuối của ứng dụng, không phải ngày phát hành hoặc ngày thu thập dữ liệu.

In [6]:
rows_before = len(apps)
missing_before = apps["Last Updated"].isna().sum()

# Định dạng gốc, ví dụ: January 7, 2018.
last_updated = pd.to_datetime(
    apps["Last Updated"],
    format="%B %d, %Y",
    errors="raise"
).astype("datetime64[ns]")

apps["Last Updated"] = last_updated

# Kiểm tra sau chuyển đổi.
print(f"Số dòng trước: {rows_before:,}")
print(f"Số dòng sau: {len(apps):,}")
print(f"Kiểu dữ liệu: {apps['Last Updated'].dtype}")
print(f"Số giá trị thiếu trước: {missing_before:,}")
print(f"Số giá trị thiếu sau: {apps['Last Updated'].isna().sum():,}")
print(f"Ngày nhỏ nhất: {apps['Last Updated'].min():%d/%m/%Y}")
print(f"Ngày lớn nhất: {apps['Last Updated'].max():%d/%m/%Y}")

display(apps[["App", "Last Updated"]].head())

Số dòng trước: 10,357
Số dòng sau: 10,357
Kiểu dữ liệu: datetime64[ns]
Số giá trị thiếu trước: 0
Số giá trị thiếu sau: 0
Ngày nhỏ nhất: 21/05/2010
Ngày lớn nhất: 08/08/2018


,App,Last Updated
0,Photo Editor & Candy Camera & Grid & ScrapBook,2018-01-07
1,Coloring book moana,2018-01-15
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",2018-08-01
3,Sketch - Draw & Paint,2018-06-08
4,Pixel Draw - Number Art Coloring Book,2018-06-20


Cột `Last Updated` đã được chuyển sang kiểu `datetime64[ns]`, không phát sinh giá trị thiếu. Ngày cập nhật nhỏ nhất là 21/05/2010, lớn nhất là 08/08/2018. Số dòng giữ nguyên ở 10.357.

**7. Xử lý `Type` thiếu và kiểm tra với `Price`**

In [7]:
rows_before = len(apps)

# ---------- 1. Nhận diện Type thiếu ----------
type_text = apps["Type"].astype("string").str.strip()
missing_type = (
    type_text.isna()
    | type_text.eq("").fillna(False)
)

print("Các bản ghi thiếu Type trước xử lý:")
display(
    apps.loc[missing_type, ["App", "Type", "Price", "Installs"]]
)

# ---------- 2. Suy ra Type từ giá hợp lệ ----------
infer_free = missing_type & apps["Price"].eq(0)
infer_paid = missing_type & apps["Price"].gt(0)

apps.loc[infer_free, "Type"] = "Free"
apps.loc[infer_paid, "Type"] = "Paid"

# ---------- 3. Kiểm tra sau xử lý ----------
type_after = apps["Type"].astype("string").str.strip()

missing_after = (
    type_after.isna()
    | type_after.eq("").fillna(False)
)

invalid_type = (
    ~missing_after
    & ~type_after.isin(["Free", "Paid"])
)

conflict = (
    (type_after.eq("Free") & apps["Price"].gt(0))
    | (type_after.eq("Paid") & apps["Price"].eq(0))
).fillna(False)

display(pd.Series({
    "Missing Type before": missing_type.sum(),
    "Inferred Free": infer_free.sum(),
    "Inferred Paid": infer_paid.sum(),
    "Missing Type after": missing_after.sum(),
    "Invalid Type labels": invalid_type.sum(),
    "Type-Price conflicts": conflict.sum()
}))

print("Các bản ghi có nhãn không hợp lệ hoặc mâu thuẫn với giá:")
display(
    apps.loc[
        invalid_type | conflict,
        ["App", "Type", "Price"]
    ]
)

print(f"Số dòng trước: {rows_before:,}")
print(f"Số dòng sau: {len(apps):,}")

Các bản ghi thiếu Type trước xử lý:


,App,Type,Price,Installs
9148,Command & Conquer: Rivals,NaN,0.0,0


Missing Type before     1
Inferred Free           1
Inferred Paid           0
Missing Type after      0
Invalid Type labels     0
Type-Price conflicts    0
dtype: int64

Các bản ghi có nhãn không hợp lệ hoặc mâu thuẫn với giá:


,App,Type,Price


Số dòng trước: 10,357
Số dòng sau: 10,357


Bản ghi "Command & Conquer: Rivals" thiếu `Type` và có `Price` = 0, nên nhóm suy ra nhãn Free. Sau xử lý, cột `Type` không còn giá trị thiếu hoặc nhãn không hợp lệ. Nhóm cũng không phát hiện thêm bản ghi có `Type` mâu thuẫn với `Price`. Do đó số dòng được giữ nguyên.

Các giá trị thiếu của `Rating`, `Current Ver` và `Android Ver` được giữ trong bảng chung. Việc lọc theo điểm đánh giá hoặc tạo nhóm `Unknown` sẽ được thực hiện riêng khi câu hỏi cần đến.

**8. Chuyển kiểu các cột phân loại và văn bản**  

Ở phần data exploration, nhóm có một số đề xuất về việc chuyển đổi các cột biểu diễn nhãn sang kiểu `category` và các cột văn bản sang kiểu `string`. Mỗi tổ hợp trong `Genres` tiếp tục được giữ như một nhãn riêng. Cụ thể:    
- `Category`, `Type`, `Content Rating`, `Genres` → `category`
- `App`, `Size`, `Current Ver`, `Android Ver` → `string`

Giữ nguyên các nhãn tổ hợp, kể cả `Education;Education`; chưa gộp nhãn hiếm hoặc chuẩn hóa các thành phần trong bước tiền xử lý chung.

In [8]:
rows_before = len(apps)

categorical_cols = ["Category", "Type", "Content Rating", "Genres"]
string_cols = ["App", "Size", "Current Ver", "Android Ver"]
converted_cols = categorical_cols + string_cols

# ---------- 1. Ghi nhận trạng thái trước chuyển đổi ----------
dtype_before = apps[converted_cols].dtypes.astype(str)
missing_before = apps[converted_cols].isna().sum()
distinct_before = apps[converted_cols].nunique()

# ---------- 2. Chuyển kiểu dữ liệu ----------
for col in categorical_cols:
    apps[col] = apps[col].astype("category")

for col in string_cols:
    apps[col] = apps[col].astype("string")

# ---------- 3. Đối chiếu trước và sau ----------
conversion_summary = pd.DataFrame({
    "Dtype before": dtype_before,
    "Dtype after": apps[converted_cols].dtypes.astype(str),
    "Missing before": missing_before,
    "Missing after": apps[converted_cols].isna().sum(),
    "Distinct before": distinct_before,
    "Distinct after": apps[converted_cols].nunique()
})

display(conversion_summary)

print(f"Số dòng trước: {rows_before:,}")
print(f"Số dòng sau: {len(apps):,}")

,Dtype before,Dtype after,Missing before,Missing after,Distinct before,Distinct after
Category,object,category,0,0,33,33
Type,object,category,0,0,2,2
Content Rating,object,category,0,0,6,6
Genres,object,category,0,0,119,119
App,object,string,0,0,9659,9659
Size,object,string,0,0,461,461
Current Ver,object,string,8,8,2831,2831
Android Ver,object,string,2,2,33,33


Số dòng trước: 10,357
Số dòng sau: 10,357


**9. Kiểm tra lại cấu trúc dữ liệu của các cột**

In [9]:
apps = apps.reset_index(drop=True)
apps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10357 entries, 0 to 10356
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   App             10357 non-null  string        
 1   Category        10357 non-null  category      
 2   Rating          8892 non-null   float64       
 3   Reviews         10357 non-null  Int64         
 4   Size            10357 non-null  string        
 5   Installs        10357 non-null  Int64         
 6   Type            10357 non-null  category      
 7   Price           10357 non-null  float64       
 8   Content Rating  10357 non-null  category      
 9   Genres          10357 non-null  category      
 10  Last Updated    10357 non-null  datetime64[ns]
 11  Current Ver     10349 non-null  string        
 12  Android Ver     10355 non-null  string        
 13  Size_MB         8831 non-null   float64       
 14  Size_varies     10357 non-null  boolean       
dtypes:

Sau tiền xử lý, bảng có 10.357 dòng và 15 cột, gồm 13 cột gốc và hai cột mới `Size_MB`, `Size_varies`. Các cột số, phân loại, văn bản và ngày tháng đã có kiểu dữ liệu phù hợp với kế hoạch